In [0]:
%run ./01_extracao_pedidos

In [0]:
from pyspark.sql.types import TimestampNTZType

COLUNAS_DESTINO = [
    "id_pedido", "id_cliente", "id_endereco_entrega",
    "dt_pedido", "status_pedido", "valor_total", "valor_frete",
    "metodo_pagamento", "dt_previsao_entrega", "dt_ultima_atualizacao_status",
    "dt_lote",
]

faltando = [c for c in COLUNAS_DESTINO if c not in df_pedidos.columns]
if faltando:
    raise ValueError(f"Colunas ausentes na origem: {faltando}")

df_carga = df_pedidos.select([
    F.col(c).cast("timestamp").alias(c)
    if isinstance(df_pedidos.schema[c].dataType, TimestampNTZType) else F.col(c)
    for c in COLUNAS_DESTINO
])

linhas_origem = df_carga.count()
print(f"Origem: {linhas_origem} linhas x {len(df_carga.columns)} colunas")
df_carga.printSchema()

In [0]:
problemas = []

if linhas_origem == 0:
    problemas.append("DataFrame de origem esta vazio")

nulos_pk = df_carga.filter(F.col(CHAVE_PRIMARIA).isNull()).count()
if nulos_pk:
    problemas.append(f"{nulos_pk} nulos na chave '{CHAVE_PRIMARIA}'")

dup_pk = linhas_origem - df_carga.select(CHAVE_PRIMARIA).distinct().count()
if dup_pk:
    problemas.append(f"{dup_pk} duplicatas na chave '{CHAVE_PRIMARIA}'")

if problemas:
    raise ValueError("Carga interrompida:\n" + "\n".join(f"  - {p}" for p in problemas))

print("[ok] origem validada")

In [0]:
PERMITIR_REDUCAO = True

try:
    antes = (
        spark.read.format("sqlserver")
             .options(**opcoes_sqlserver(TABELA_DESTINO))
             .load()
             .count()
    )
    print(f"{TABELA_DESTINO} existe com {antes} linhas; origem tem {linhas_origem}.")
except Exception as e:
    antes = None
    print(f"{TABELA_DESTINO} ainda nao existe (ou esta inacessivel) - sera criada.")
    print(f"Detalhe: {type(e).__name__}")

if antes is not None and antes > linhas_origem and not PERMITIR_REDUCAO:
    raise RuntimeError(
        f"O overwrite reduziria {TABELA_DESTINO} de {antes} para {linhas_origem} linhas.\n"
        "Se for intencional, defina PERMITIR_REDUCAO = True nesta celula e reexecute."
    )

In [0]:
import time

inicio = time.time()
(
    df_carga.write
            .format("sqlserver")
            .options(**opcoes_sqlserver(TABELA_DESTINO))
            .mode(MODO_ESCRITA)
            .save()
)
print(f"[ok] gravacao concluida em {time.time() - inicio:.1f}s -> {TABELA_DESTINO} ({MODO_ESCRITA})")

In [0]:
df_destino = (
    spark.read.format("sqlserver")
         .options(**opcoes_sqlserver(TABELA_DESTINO))
         .load()
)


def assinatura(df):
    return df.agg(
        F.count("*").alias("linhas"),
        F.countDistinct(CHAVE_PRIMARIA).alias("chaves"),
        F.round(F.sum("valor_total"), 2).alias("soma_valor_total"),
    ).first().asDict()


origem, destino = assinatura(df_carga), assinatura(df_destino)
for k in origem:
    marca = "ok" if origem[k] == destino[k] else "!!"
    print(f"[{marca}] {k:<17} origem={origem[k]}  destino={destino[k]}")

print("\nStatus:", "SUCESSO - origem e destino batem." if origem == destino else "ATENCAO - divergencia, investigar.")
df_destino.printSchema()
display(df_destino.orderBy("dt_pedido").limit(10))